In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from scipy.stats import binned_statistic

In [ ]:
N_SIGMA = 1.0  # outlier threshold

unclean = pd.read_parquet(
    "../results_main/catalogs/default/galaxy_catalog_background.parquet"
)
clean = unclean.copy()

colors = ["g-i", "g-r", "r-i", "i-z"]

dz = 0.05
z_bins = np.arange(unclean.z_phot.min() - 2 * dz, unclean.z_phot.max() + 2 * dz, dz)
bin_centers = 0.5 * (z_bins[1:] + z_bins[:-1])

outlier_mask = np.zeros(len(unclean), dtype=bool)

for color in colors:
    # Running median as a function of redshift
    median_stat, _, _ = binned_statistic(
        unclean.z_phot.values,
        unclean[color].values,
        statistic=np.nanmedian,
        bins=z_bins,
    )
    # Interpolate to each galaxy's redshift and detrend
    median_interp = np.interp(unclean.z_phot.values, bin_centers, median_stat)
    detrended = (
        unclean[color].values
        - median_interp
        + 1.0  # np.nanmedian(unclean[color].values)
    )
    clean[f"{color}"] = detrended

    # Robust sigma via MAD (scaled to match Gaussian sigma), also as a function of redshift
    mad_stat, _, _ = binned_statistic(
        unclean.z_phot.values,
        np.abs(detrended),
        statistic=np.nanmedian,
        bins=z_bins,
    )
    sigma_interp = 1.4826 * np.interp(unclean.z_phot.values, bin_centers, mad_stat)

    outlier_mask |= np.abs(detrended) > N_SIGMA * sigma_interp

clean = clean[~outlier_mask].copy()
print(
    f"Removed {outlier_mask.sum():,} / {len(unclean):,} galaxies ({100 * outlier_mask.mean():.1f}%)"
)

In [ ]:
xlim = (-0.2, 2.2)
ylim = (0.7, 1.4)


def plot_vs_z(ax, data, col, label_left, contours=False):

    if not contours:
        # Plot hexbin
        d = 0.02
        gridsize = np.abs(np.diff((xlim, ylim)) / d).astype(int).ravel()
        ax.hexbin(
            data[col].values,
            data.z_phot.values,
            norm="log",
            edgecolors="none",
            extent=(*xlim, *ylim),
            gridsize=tuple(gridsize),
            rasterized=True,
        )

    if contours:
        # First plot scatter to show outliers
        ax.scatter(
            data[col].values,
            data.z_phot.values,
            s=0.1,
            color="C0",
            edgecolors="none",
            zorder=-1,
            rasterized=True,
        )

        # Density contour plot with logarithmic color scale
        range = ([[-2, 2], [0.6, 1.5]],)
        top_level = (200,)
        vmax = (100,)
        counts, xedges, yedges = np.histogram2d(
            data[col].values,
            data.z_phot.values,
            bins=100,
            range=range,
        )
        counts = np.where(counts > 1, counts, np.nan)
        X, Y = np.meshgrid(
            0.5 * (xedges[:-1] + xedges[1:]),
            0.5 * (yedges[:-1] + yedges[1:]),
        )
        ax.contourf(
            X,
            Y,
            counts.T,
            levels=np.linspace(1, top_level, 100),
            cmap="viridis",
            norm=mcolors.LogNorm(vmin=1, vmax=vmax),
        )
    if label_left:
        label_settings = dict(ha="left", x=0.08)
    else:
        label_settings = dict(ha="right", x=0.92)
    sig = np.nanstd(data[col].values)
    if sig < 10:
        label_settings["s"] = f"$\\sigma = {sig:.2f}$"
    else:
        label_settings["s"] = f"$\\sigma = {sig:.0f}$"
    ax.text(
        **label_settings,
        y=0.92,
        va="top",
        transform=ax.transAxes,
        fontsize=8,
        bbox=dict(
            facecolor="white",
            alpha=1.0,
            edgecolor="none",
            boxstyle="round",
            pad=0.1,
        ),
    )


def plot_before_vs_after(ax1, ax2, col, label):
    # Plot original contours
    plot_vs_z(ax1, unclean, col, label_left=False)

    # Running median
    dz = 0.05
    bins = np.arange(unclean.z_phot.min() - 2 * dz, unclean.z_phot.max() + 2 * dz, dz)
    stat, bin_edges, _ = binned_statistic(
        unclean.z_phot.values,
        unclean[col].values,
        statistic=np.nanmedian,
        bins=bins,
    )
    bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
    ax1.plot(stat, bin_centers, color="r", lw=1, label="Median")
    ax2.set(xlabel=label)

    # Plot cleaned contours
    label_left = False if col == "g-i" else False
    plot_vs_z(ax2, clean, col, label_left=label_left)


# Colors
fig, axes = plt.subplots(
    2,
    4,
    figsize=(7, 2 * 7 / 4),
    dpi=200,
    sharex="col",
    sharey="row",
)

for i, color in enumerate(["g-i", "g-r", "r-i", "i-z"]):
    label = "$" + color + "$"
    plot_before_vs_after(*axes[:, i], color, label)

fig.subplots_adjust(wspace=0, hspace=0.05)
for ax in axes[:, 0]:
    ax.set(
        ylabel="$z_{\\mathrm{phot}}$",
        yticks=np.arange(0.6, 1.6, 0.2),
    )
for ax in axes.flatten():
    ax.set(xlim=xlim, ylim=ylim, xticks=np.arange(0, 2.5, 0.5))
    # ax.set_xticks(np.arange(-0.0, 2, 1), minor=True)
    ax.tick_params(which="both", top=True, right=True, direction="in", width=0.5)
    ax.tick_params(which="major", length=3)
    ax.tick_params(which="minor", length=1.5)

fig.savefig("../figures/fig_cleaning.pdf", bbox_inches="tight")